In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

import flax.linen as nn

import znnl as nl

# Linalg help
import flax.linen as nn
import jax

# system and plotting help
from rich.progress import track


In [ ]:
data_path = "/data/stovey/surrogate-variables/experiment_data.npy"

raw_data = np.load(data_path, allow_pickle=True)
processed_data = np.load("experiment-analysis.npy", allow_pickle=True)

# Analysis tools

In [ ]:
@dataclass
class Experiment:
    # Experiment Parameters
    loss_fn: str
    width: int
    depth: int
    learning_rate: float
    optimizer: str
    epochs: int
    batch_size: int
    ds_size: int

    # Results
    parameters: list
    train_loss: np.ndarray


@dataclass
class Measurement:
    width: int
    depth: int
    loss: np.ndarray
    trace: np.ndarray
    entropy: np.ndarray
    representations: np.ndarray
    loss_derivatives: np.ndarray
    class_entropy: dict

In [ ]:
def linear_boundary(
    data: np.ndarray, gradient: float = -1.0, intercept: float = 1.0
):
    """
    Create a linear boundary between classes.
    
    Parameters
    ----------
    data : np.ndarray (n_samples, 2)
            Data to be converted into classes.
    gradient : float (default = 1.0)
            Gradient of the line, default 1.0.
    intercept : float (default = 1.0)
            Intercept of the line, default 1.0.
    """
    # y = m * x + c
    reference_values = gradient * data[:, 0] + intercept

    differences = data[:, 1] - reference_values
    differences[differences > 0] = 1
    differences[differences < 0.] = 0

    return differences

class Generator(nl.data.DataGenerator):
    def __init__(self, n_samples: int, discriminator: callable = linear_boundary):
        """
        Instantiate the class.
       
        Parameters
        ----------
        n_samples : int
                Number of samples to generate per class.
        discriminator : callable
                Function to map x, y values to a class.
        """
        # Create the data-sets
        train_data = np.array(jax.random.uniform(
            jax.random.PRNGKey(0), minval=0., maxval=1., shape=(1000, 2)
        ))
        train_data = np.clip(train_data, 0, 1)
        train_targets = discriminator(train_data)  # build classes (0, 1)

        class_one_indices = np.where(
            train_targets == 0
        )[0][:n_samples]

        class_two_indices = np.where(
            train_targets == 1
        )[0][:n_samples]

        indices = np.hstack((class_one_indices, class_two_indices))
        np.random.shuffle(indices)

        self.train_ds = {
            "inputs": np.take(train_data, indices, axis=0),
            "targets": np.take(np.array(jax.nn.one_hot(train_targets, num_classes=2)), indices, axis=0)
        }


In [ ]:
def build_network(width: int, depth: int):
    """
    Construct a flax model with seperate layers.
    
    Parameters
    ----------
    width : int
            How many unit in hidden layers.
    depth : int
            How many hidden layers.
    """
    # Template of a layer.
    class HiddenLayer(nn.Module):
        @nn.compact
        def __call__(self, x):
            """
            Call method for hidden layer.
            """
            x = nn.Dense(width)(x)

            return x

    class Model(nn.Module):
        def setup(self):
            self.hidden_layers = [
                HiddenLayer() for _ in range(depth)
            ]

        @nn.compact
        def __call__(self, x):
            """
            Call method for entire model.
            """
            intermediate_representations = []
            for item in self.hidden_layers:
                x = item(x)
                intermediate_representations.append(x)
                x = nn.relu(x)
                
            # Output layer
            x = nn.Dense(2)(x)

            return x, intermediate_representations

    return Model


In [ ]:
def compute_msd(data: np.ndarray):
    """
    Compute the msd of single layer data.
    
    I take an average over dimensions and points.
    
    Parameters
    ----------
    data : np.ndarray (n_time_steps, n_points, dimension)
    
    Returns
    -------
    msd : np.ndarray (n_timesteps, 1)
            MSD of the data.
    """
    
    msd = (
        (data - data[0, :, :]) ** 2
    ).mean(axis=1).mean(axis=-1)
    
    return msd

# One Layer

In [ ]:
@dataclass
class Measurement:
    width: int
    depth: int
    loss: np.ndarray
    trace: np.ndarray
    entropy: np.ndarray
    representations: np.ndarray
    loss_derivatives: np.ndarray
    class_entropy: dict


In [ ]:
analysed_data = processed_data[0]
network_data = raw_data[0]

generator = Generator(n_samples=50)

In [ ]:
network = build_network(width=network_data.width, depth=network_data.depth)()

In [ ]:
output_representations = []
hidden_representations = []

for params in network_data.parameters:
    outs, reps = network.apply(
        {"params": params}, 
        generator.train_ds["inputs"]
    )
    output_representations.append(outs)
    hidden_representations.append(reps[0])  # 0 due to only having one layer

output_representations = np.array(output_representations)
hidden_representations = np.array(hidden_representations)

In [ ]:
hidden_msd = compute_msd(hidden_representations)
output_msd = compute_msd(output_representations)
trace = analysed_data.trace

In [ ]:
fig, ax_1 = plt.subplots()
ax_2 = ax_1.twinx()

ax_1.plot(hidden_msd, ls="--")
# ax_1.plot(np.convolve(hidden_msd , output_msd, mode="same"), ls="--")

# ax_1.plot(output_msd, ls="--")
ax_2.plot(trace / 100)

ax_1.set_yscale("log")
ax_2.set_yscale("log")
ax_1.set_xscale("log")

plt.show()

In [ ]:
out = np.convolve(hidden_msd , output_msd, mode="valid")

In [ ]:
out.shape